# Text Preprocessing & Tokenization — AI Lab Instructor Notebook

*CV & NLP Basics · Hard*



5 tasks, 1 left as exercises (no source solution).

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import re
import unicodedata
from collections import Counter

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')


## Task 1: Dataset & Setup
*Section: Setup*

# Text Preprocessing & Tokenization — Student Lab

We implement robust text normalization and tokenization from scratch (no external datasets).

Focus areas:
- normalization (unicode, casing, whitespace)
- word tokenization (punctuation as tokens)
- character tokenization
- a toy BPE-style subword tokenizer

## Corpus (offline)
We intentionally include messy text: unicode quotes, dashes, emojis, accents, URLs, weird spacing, and a few strings where NFKC effects are very visible.

In [ ]:
import re
import unicodedata
from collections import Counter

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

corpus = [
    "Hello, world!",
    "Email me at foo.bar+baz@example.com — thanks!",
    "I can’t believe it’s not butter…",
    "Café prices: $3.50, $4.00, $4.25",
    "Visit https://example.com/docs?id=123&ref=abc",
    "   multiple   spaces\n and\t tabs   ",
    "naïve façade coöperate",
    "Numbers: 1,000 vs 1000; 3.14159; 2e10",
    "Emoji 🙂 and symbols ©™ should not crash",
    'Quotes: “smart” vs "dumb"; dashes: – — -',
    "Ｆｕｌｌｗｉｄｔｈ ＡＢＣ １２３",
    "Steps ① ② ③ should normalize cleanly",
    "Office ﬂoor near Ⅳ street",
    "Ligatures: oﬃce, coeﬃcient, and suﬃx",
]
print('n_docs', len(corpus))
for i, s in enumerate(corpus):
    print(i, repr(s))

**Why this works.** # Text Preprocessing & Tokenization — Instructor Lab

Solutions + short rationales. Fully runnable top-to-bottom.

## Normalization
Rationale: `NFKC` reduces unicode variants; whitespace collapse makes downstream tokenization deterministic.

## Task 2: Implement `normalize_text(s)` with:
*Section: Normalization*

## Section 0 — Normalization

### Task 0.1
Implement `normalize_text(s)` with:
- Unicode normalization (NFKC)
- Optional lowercasing
- Whitespace normalization (collapse runs to single spaces)

Regex note:
- `\s` means any whitespace character: space `' '`, newline `'\n'`, tab `'\t'`, carriage return `'\r'`
- so `r'\s+'` means one or more whitespace characters in a row

FAANG gotcha: normalization decisions are task-dependent; don’t blindly lowercase in all problems.

In [ ]:
_WS_RE = re.compile(r'\s+')

def normalize_text(s: str, *, lowercase: bool = True) -> str:
    s2 = unicodedata.normalize('NFKC', s)
    if lowercase:
        s2 = s2.lower()
    s2 = _WS_RE.sub(' ', s2).strip()
    return s2

check('ws_collapse', normalize_text('  A\nB\tC  ') == 'a b c')
for i, s in enumerate(corpus):
    print(i, repr(s))
for i in range(len(corpus)):
    print(normalize_text(corpus[i]))

In [ ]:
# Checks
check('ws_collapse', normalize_text('  A\nB\tC  ') == 'a b c')

**Why this works.** ## Normalization
Rationale: `NFKC` reduces unicode variants; whitespace collapse makes downstream tokenization deterministic.

## Task 3: Implement `tokenize_words(s)` using a regex that produces tokens including punctuation.
*Section: Word tokenization*

## Section 1 — Word tokenization

We want punctuation separated as tokens (e.g., commas, periods, parentheses).

### Task 1.1
Implement `tokenize_words(s)` using a regex that produces tokens including punctuation.

Design constraints:
- deterministic
- does not drop punctuation
- does not crash on unicode

Note: This is a *toy* tokenizer; real-world tokenizers are more nuanced.

In [ ]:
_TOK_RE = re.compile(
    r"""
    (?:[a-zA-Z]+(?:'[a-zA-Z]+)?)
    |(?:\d+(?:,\d{3})*(?:\.\d+)?(?:e[+-]?\d+)?)
    |(?:[^\s])
    """,
    re.VERBOSE
)

def tokenize_words(s: str) -> list[str]:
    return _TOK_RE.findall(normalize_text(s, lowercase=True))

check('punct', ',' in tokenize_words('hi, there'))
check('num', '3.14159' in tokenize_words('pi=3.14159'))
for i, s in enumerate(corpus):
    print(i, repr(s))
    print(tokenize_words(s))
print(tokenize_words(corpus[1]))

In [ ]:
# Checks
check('punct', ',' in tokenize_words('hi, there'))
check('num', '3.14159' in tokenize_words('pi=3.14159'))

**Why this works.** ## Word tokenization
Rationale: keep punctuation tokens so you can reconstruct structure (e.g., sentence boundaries) and avoid information loss.

## Task 4: Implement `tokenize_chars` and `detokenize_chars`.
*Section: Character tokenization*

## Section 2 — Character tokenization

### Task 2.1
Implement `tokenize_chars` and `detokenize_chars`.
We will keep spaces as a token to preserve exact reversibility.

In [ ]:
def tokenize_chars(s: str) -> list[str]:
    return list(normalize_text(s, lowercase=False))

def detokenize_chars(tokens: list[str]) -> str:
    return ''.join(tokens)

s = 'Hið'
for i, doc in enumerate(corpus):
    print(i, repr(doc))
check('roundtrip', detokenize_chars(tokenize_chars(s)) == normalize_text(s, lowercase=False))

In [ ]:
# Checks
check('roundtrip', detokenize_chars(tokenize_chars(s)) == normalize_text(s, lowercase=False))

**Why this works.** ## Char tokenization
Rationale: char tokenization has no OOV but can be long sequences.

## Task 5: Implement the helper functions used to learn merges and to encode words.
*Section: Subword tokenization (BPE-style)*

## Section 3 — Subword tokenization (BPE-style)

We implement a toy BPE algorithm:
- represent each word as characters plus an end-of-word marker `</w>`
- repeatedly merge the most frequent adjacent pair

### Task 3.1
Implement the helper functions used to learn merges and to encode words.

In [ ]:
# No source solution — left as an exercise.